Make color cutouts from DP2 deep coadd images.

Based on DP2 tutorials 103.5, 103.7, 202.1, Padma's notebook.

Please run it on the [RSP](https://data.lsst.cloud). 

In [ ]:
import lsst.afw.display as afw_display
from lsst.images.serialization import read_archive
from lsst.rsp import RSPDiscovery
from lsst.rsp.utils import get_pyvo_auth

from pyvo.dal.adhoc import SodaQuery

import io
import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import AsinhStretch, ImageNormalize, make_lupton_rgb
from astropy import units as u

%matplotlib inline

In [ ]:
def get_dl_result(band, ra, dec, radius):

    circle = (ra, dec, radius)

    results = sia_client.search(pos=circle, calib_level=3,
                                dpsubtype='lsst.deep_coadd')
    print("Num of sia_client search result: ", len(results))
    #t = results.to_table()
    #t.pprint_all()

    band_arr = results['lsst_band']

    try:
        index = int(np.where(band_arr == band)[-1][-1])
    except:
        print(f'Band {band} does not exist!')
        return None

    dl_result = discovery.get_datalink_results(results[index])
    
    print(f"Datalink status: {dl_result.status}.")

    return dl_result
    

def get_cutout(band, ra, dec, radius=0.002):

    dl_result = get_dl_result(band, ra, dec, radius)

    if dl_result is None:
        return None

    sq = SodaQuery.from_resource(dl_result,
                                 dl_result.get_adhocservice_by_id("cutout-sync"),
                                 session=get_pyvo_auth())

    sq.circle = (ra * u.deg, dec * u.deg, radius * u.deg)

    cutout_bytes = sq.execute_stream().read()
    sq.raise_if_error()

    cutout = read_archive(io.BytesIO(cutout_bytes))

    return cutout

In [ ]:
def normalize_band(image, asinh_a=0.002):
    
    data = image.array

    vmin = -0.03
    #vmax = 200
    vmax = 3000

    norm = ImageNormalize(vmin=vmin, vmax=vmax,
                          stretch=AsinhStretch(a=asinh_a),
                          clip=True,
                         )

    scaled = norm(data)
    return scaled


def combine_RGB(R_image, G_image, B_image):
    
    R_channel = normalize_band(R_image)
    G_channel = normalize_band(G_image)
    B_channel = normalize_band(B_image)

    RGB_image = np.dstack([R_channel, G_channel, B_channel])

    return RGB_image

    

In [ ]:
def plot_afw(image):

    fig, ax = plt.subplots(figsize=(6,6))
    display = afw_display.Display(frame=fig)
    #display.scale('linear', 'zscale')
    display.scale('linear', -75, 125)
    display.image(image)

    return 0

In [ ]:
def plot_RGB(R_image, G_image, B_image, make_lupton=False, lim=None):

    if R_image is None or G_image is None or B_image is None:
        return 1

    RGB_image = combine_RGB(R_image, G_image, B_image)
    if make_lupton:
        RGB_image = make_lupton_rgb(R_image.array,
                                    G_image.array,
                                    B_image.array,
                                    stretch=0.002, Q=0.001)

    if lim is not None:
        xmin, xmax, ymin, ymax = lim
        RGB_image = RGB_image[xmin:xmax, ymin:ymax]
    
    fig = plt.figure(figsize=(6,6))
    im = plt.imshow(RGB_image,
                    origin='lower')

    return 0

In [ ]:
afw_display.setDefaultBackend("matplotlib")

discovery = RSPDiscovery("dp2")
sia_client = discovery.get_sia_client()

In [ ]:
# DESJ0407-5006
#target_ra, target_dec = 61.792580, -50.100250

# CXCOJ100201.50+020330.0
target_ra, target_dec = 150.506300, 2.058100

In [ ]:
cutout_u = get_cutout('u', target_ra, target_dec)
cutout_g = get_cutout('g', target_ra, target_dec)
cutout_r = get_cutout('r', target_ra, target_dec)
cutout_i = get_cutout('i', target_ra, target_dec)
cutout_z = get_cutout('z', target_ra, target_dec)
cutout_y = get_cutout('y', target_ra, target_dec)

In [ ]:
#plot_afw(cutout_g)

In [ ]:
plot_RGB(cutout_r, cutout_g, cutout_u)
plot_RGB(cutout_i, cutout_r, cutout_g)
plot_RGB(cutout_z, cutout_i, cutout_r)
plot_RGB(cutout_y, cutout_z, cutout_i)

In [ ]:
plot_RGB(cutout_r, cutout_g, cutout_u, True)
plot_RGB(cutout_i, cutout_r, cutout_g, True)
plot_RGB(cutout_z, cutout_i, cutout_r, True)
plot_RGB(cutout_y, cutout_z, cutout_i, True)

In [ ]:
plot_RGB(cutout_y, cutout_r, cutout_g, True)
plot_RGB(cutout_z, cutout_g, cutout_u, True)